<a href="https://colab.research.google.com/github/somendrew/GenZ-LM/blob/main/1.FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Setup

In [ ]:
!pip install -q  peft trl bitsandbytes accelerate
!pip install -q wandb  #for logging

In [ ]:
import torch

In [ ]:
from google.colab import userdata
HF_Token = userdata.get('HF_Token')

## Step 2&3: Import DataSet and Prepare dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/data (1).jsonl")
dataset = dataset["train"]  # ✅ extract from DatasetDict

print(f"Total examples: {len(dataset)}")
print(dataset[0])

# ================================
# Step 2: Format as Messages
# ================================
def format_as_messages(example):
    # ✅ Handle empty input field (many rows have input: "")
    if example["input"] and example["input"].strip():
        user_content = f"{example['instruction']}\n\n{example['input']}"
    else:
        user_content = example["instruction"]  # skip input if empty

    return {
        "messages": [
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": example["output"]}
        ]
    }

dataset = dataset.map(format_as_messages, remove_columns=dataset.column_names)

# Sanity check
print(f"\nFormatted dataset size: {len(dataset)}")
print(dataset[0])
print(dataset[1])  # this one has empty input

## Step 4: Load the Base Model in 4-bit

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, #save extra memory
)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

## Step 5: Configure Lora Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#prepare model for qlora
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[         # which layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


## Step 6: Train with TRL's SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./genz-slang-model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    weight_decay=0.001,
    bf16=True,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    eval_strategy="no",
    report_to="none",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,3.567726
20,1.736011
30,1.316345
40,0.825485


TrainOutput(global_step=48, training_loss=1.668426086505254, metrics={'train_runtime': 2352.4507, 'train_samples_per_second': 0.307, 'train_steps_per_second': 0.02, 'total_flos': 3773742024966144.0, 'train_loss': 1.668426086505254})

## Step 7: Save

In [ ]:
# Save just the LoRA adapter (lightweight, ~50–100MB)
trainer.model.save_pretrained("./genz-lora-adapter")
tokenizer.save_pretrained("./genz-lora-adapter")
print("✅ Model saved!")


✅ Model saved!


### Downloading model: To avoid session crash out

In [ ]:
!zip -r lora-adapter.zip /content/genz-lora-adapter
!zip -r slang-model.zip /content/genz-slang-model

  adding: content/genz-lora-adapter/ (stored 0%)
  adding: content/genz-lora-adapter/tokenizer_config.json (deflated 48%)
  adding: content/genz-lora-adapter/README.md (deflated 65%)
  adding: content/genz-lora-adapter/tokenizer.json (deflated 85%)
  adding: content/genz-lora-adapter/chat_template.jinja (deflated 64%)
  adding: content/genz-lora-adapter/adapter_config.json (deflated 58%)
  adding: content/genz-lora-adapter/adapter_model.safetensors (deflated 21%)
  adding: content/genz-slang-model/ (stored 0%)
  adding: content/genz-slang-model/README.md (deflated 44%)
  adding: content/genz-slang-model/checkpoint-48/ (stored 0%)
  adding: content/genz-slang-model/checkpoint-48/trainer_state.json (deflated 63%)
  adding: content/genz-slang-model/checkpoint-48/scheduler.pt (deflated 62%)
  adding: content/genz-slang-model/checkpoint-48/tokenizer_config.json (deflated 48%)
  adding: content/genz-slang-model/checkpoint-48/training_args.bin (deflated 53%)
  adding: content/genz-slang-model